In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

#Visualization
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

#Settings
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 1

import warnings
warnings.filterwarnings('ignore')


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

print

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
from sklearn.preprocessing import LabelEncoder
# Load data
train_df = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')
#fill 
train_df['Age'] = df['Age'].fillna(df['Age'].mean())
test_df['Age'] = df['Age'].fillna(df['Age'].mean()) 

train_df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
test_df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
#drop missing values
train_df['Embarked'] = df['Embarked'].dropna()
test_df['Embarked'] = df['Embarked'].dropna()
#drop the cabin column
train_df = train_df.drop(columns=['Cabin','Name', 'Ticket','PassengerId'])
test_df = test_df.drop(columns=['Cabin','Name', 'Ticket','PassengerId'])

#Encode categorical variables
le = LabelEncoder()
for col in ['Sex', 'Embarked']:
    train_df[col]= le.fit_transform(train_df[col])
    test_df[col]= le.transform(test_df[col])
train_df.info()

In [ ]:
#visulize the data

#Survival count
sns.countplot(x='Survived', data =train_df)
plt.title("Survival Count (0= Dead, 1=Survived)")
plt.show()

In [ ]:
#Survival by gender
sns.countplot(x='Sex', hue='Survived', data=train_df)
plt.title("Survival by Gender")
plt.show()

In [ ]:
#survival by class
sns.countplot(x='Pclass', hue='Survived', data=train_df)
plt.title("Survival by passenger class")
plt.show()

In [ ]:
#Age distribution
sns.histplot(train_df['Age'], bins=30, kde=True)
plt.title("Age distribution")
plt.show()

In [ ]:
#Age vs survival
sns.boxplot(x='Survived', y ='Age', data= train_df)
plt.title("Age vs Survival")
plt.show()

In [ ]:
num_df = train_df.select_dtypes(include=['number'])
plt.figure(figsize=(10,6))
sns.heatmap(num_df.corr(), annot=True, cmap='coolwarm')
plt.title("Feature Correlation")
plt.show()

In [ ]:
X_train = train_df.drop('Survived', axis=1)
y_train = train_df['Survived']

X_test =test_df.copy()
#import and train 
from sklearn.ensemble import RandomForestClassifier

#model
model = RandomForestClassifier(n_estimators=100,random_state=42)

#train
model.fit(X_train,y_train)

#predict
predictions = model.predict(X_test)
survival_probs = model.predict_proba(X_test)




In [ ]:
# survival probability for class 1 (survived)
survival_probs = model.predict_proba(X_test)[:, 1]

# attach to test_df so we can visualize
X_test_viz = X_test.copy()
X_test_viz['Survival_Prob'] = survival_probs

plt.figure(figsize=(8,5))
sns.histplot(X_test_viz['Survival_Prob'], bins=20, kde=True)
plt.title("Distribution of Predictes Survival Probabilities")
plt.xlabel("Survival Probability")
plt.ylabel("Number of Passengers")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x=X_test_viz['Sex'], y=X_test_viz['Survival_Prob'])
plt.title("Survival Probability by Gender")
plt.xlabel("Sex (0=female, 1=male)")
plt.ylabel("Predicted Survival Probability")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x=X_test_viz['Pclass'], y=X_test_viz['Survival_Prob'])
plt.title("Survival Probability by Passenger Class")
plt.xlabel("Passenger Class (1=1st, 2=2nd, 3=3rd)")
plt.ylabel("Predicted Survival Probability")
plt.show()